# NCS seismic foundation model on the Norton crop (proof of concept, GPU)

Runs the open-source **NCS** foundation model (Norsk Regnesentral; Ordoñez et al. 2025, arXiv 2603.23211) on the same seismic crop as the RGT workflow, and puts its embeddings **exactly onto the seismic grid** so they can later be combined with RGT and the LFM.

1. **Inference:** `NorskRegnesentralSTI/NCS-v1-2d-base`, with the weights downloaded once into Drive → a 768-channel feature cube (Zarr, cached in Drive).
2. **Registration check:** NCS coordinates are 0-based sample indices of patch centres on the SEG-Y's sorted inline / crossline / sample axes. They're converted to inline/crossline numbers and TWT, and checked.
3. **PCA** (fitted on a random sample, applied in chunks) and **k-means facies** for several K.
4. **Similarity to the well:** cosine similarity of every embedding to the one at the well. A first look at facies-aware well interpolation for the LFM.
5. **Lateral embedding similarity:** an embedding-based discontinuity map, compared with the amplitude coherence used by the classical RGT.
6. **Outputs** on the seismic grid, as SEG-Y (facies by nearest neighbour, the rest linearly interpolated).

**What this establishes:** that the embeddings can be computed, registered and compared with the RGT tools on this data. **What it doesn't:** that the clusters are geological facies, or that embeddings improve the LFM. Those need the blind-well and horizon tests in the RGT notebooks.

**Before you start:** **Runtime → Change runtime type → L4 GPU** (A100 if available), then **Runtime → Run all**.

## 1. Settings

In [ ]:
DATA_SOURCE   = "drive"
DRIVE_FOLDER  = "/content/drive/MyDrive/PUBLIC"
DATA_FOLDER   = "/content"
SEGY_FILE     = "Full_Stack_8_41_CROP_1793889427461.sgy"

NCS_MODEL     = "NorskRegnesentralSTI/NCS-v1-2d-base"   # Hugging Face ID (only used for the first download)
MODEL_SUBDIR  = "models/NCS-v1-2d-base"                  # weights kept here inside DRIVE_FOLDER after the first download
DIRECTION     = "dir0"           # 2D model runs along inlines ("dir0") or crosslines ("dir90")
DENSIFY       = 4                # embedding every 16/DENSIFY samples: 4 → every 4 (~2.5 GB here); 8 → every 2 (~20 GB)
NCS_DTYPE     = "float16"
NCS_BATCH     = 64

WELL_IL, WELL_XL = 1123, 2098    # reference trace for sections and similarity
WELL_TWT_MS   = 1800             # reference time for the similarity map
INTERVAL_MS   = (1300, 2300)

N_PCA         = 32
PCA_SAMPLES   = 100_000
L2_NORMALISE  = True             # as in the earlier NCS work
K_VALUES      = [4, 6, 8, 12]
SEED          = 0

## 2. Set up: NCS package, Drive, seismic

In [ ]:
!pip -q install segyio "NCS[inference] @ git+https://github.com/NorskRegnesentral/NCS_models"
import os, time, shutil, sys, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, xarray as xr, torch, segyio
warnings.filterwarnings("ignore", category=RuntimeWarning)
RUN_START = time.time()
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (inference will be very slow)")

if DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    src, dst = os.path.join(DRIVE_FOLDER, SEGY_FILE), os.path.join(DATA_FOLDER, SEGY_FILE)
    if not (os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src)):
        shutil.copyfile(src, dst); print("copied", SEGY_FILE)
SEGY_PATH = os.path.join(DATA_FOLDER, SEGY_FILE)
if not os.path.exists(SEGY_PATH):
    raise FileNotFoundError(f"{SEGY_FILE} not found. Files here: {sorted(os.listdir(DATA_FOLDER))}")

!git clone -q https://github.com/fergusrssmith/RGT /content/RGT 2>/dev/null || git -C /content/RGT pull -q
sys.path[:0] = ["/content/RGT/colab", "/content/RGT/demo"]
import importlib, rgt_lfm as R, ncs_tools as N
for m in (R, N): importlib.reload(m)

BASE = DRIVE_FOLDER if DATA_SOURCE == "drive" else DATA_FOLDER
OUT_DIR = os.path.join(BASE, "outputs", time.strftime("%Y-%m-%d_%H%M", time.gmtime(RUN_START)) + "_ncs")
os.makedirs(OUT_DIR, exist_ok=True)
stem = os.path.splitext(SEGY_FILE)[0]
FEAT_PATH = os.path.join(BASE, "outputs", f"{stem}_ncs_{DIRECTION}_dens{DENSIFY}.zarr")

G = R.read_segy_cube(SEGY_PATH)
cube, ilines, xlines, samples, dt = G["cube"], G["ilines"], G["xlines"], G["samples"], G["dt"]
NI, NX, NT = cube.shape
print(f"seismic: {NI} inlines × {NX} crosslines × {NT} samples; header bytes {G['il_byte']}/{G['xl_byte']}")
if (G["il_byte"], G["xl_byte"]) != (189, 193):
    print("WARNING: NCS reads inline/crossline from bytes 189/193 (segyio default). This file uses different bytes, "
          "so its geometry may be read differently by NCS. Re-export with standard bytes.")
ci = int(np.searchsorted(ilines, WELL_IL)) if WELL_IL in ilines else NI // 2
cx = int(np.searchsorted(xlines, WELL_XL)) if WELL_XL in xlines else NX // 2
vclip = np.percentile(np.abs(cube), 99)

## 3a. Model weights: downloaded once into Drive, loaded from there afterwards

The first time, the weights are downloaded from Hugging Face into `<DRIVE_FOLDER>/models/NCS-v1-2d-base/`, and the exact version (the Hugging Face commit ID) is written to `_source.txt` there. Later sessions load them straight from Drive.
To run locally later, copy that folder to your PC and point `model_path` at it.
If Colab asks to grant access to a secret (`HF_TOKEN`), you can choose "Don't grant": this model is public.

In [ ]:
MODEL_DIR = os.path.join(BASE, MODEL_SUBDIR)
if os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    print("model weights from Drive:", MODEL_DIR)
    if os.path.exists(os.path.join(MODEL_DIR, "_source.txt")):
        print(open(os.path.join(MODEL_DIR, "_source.txt")).read())
else:
    from huggingface_hub import snapshot_download, HfApi
    tmp_model = "/content/_ncs_model_download"
    if os.path.exists(tmp_model): shutil.rmtree(tmp_model)
    t0 = time.time()
    snapshot_download(repo_id=NCS_MODEL, local_dir=tmp_model)
    try:
        sha = HfApi().model_info(NCS_MODEL).sha
    except Exception as e:
        sha = f"unknown ({type(e).__name__})"
    with open(os.path.join(tmp_model, "_source.txt"), "w") as fh:
        fh.write(f"source: https://huggingface.co/{NCS_MODEL}\ncommit: {sha}\n"
                 f"downloaded (UTC): {time.strftime('%Y-%m-%d %H:%M', time.gmtime())}\n")
    for junk in (".cache", ".gitattributes"):
        jp = os.path.join(tmp_model, junk)
        if os.path.isdir(jp): shutil.rmtree(jp)
        elif os.path.isfile(jp): os.remove(jp)
    os.makedirs(os.path.dirname(MODEL_DIR), exist_ok=True)
    shutil.copytree(tmp_model, MODEL_DIR, dirs_exist_ok=True)
    print(f"downloaded {NCS_MODEL} (commit {sha}) in {time.time()-t0:.0f} s and saved to {MODEL_DIR}")
print("files:", sorted(f for f in os.listdir(MODEL_DIR) if not f.startswith(".")))

## 3. Run NCS (cached in Drive)

Skipped if the feature cube already exists in `PUBLIC/outputs/`. Expect several minutes on an L4.

In [ ]:
if os.path.exists(FEAT_PATH):
    print("feature cube already in Drive:", FEAT_PATH)
else:
    from NCS.inference import run_inference
    tmp = f"/content/{os.path.basename(FEAT_PATH)}"          # write locally (fast), then copy to Drive
    if os.path.exists(tmp): shutil.rmtree(tmp)
    t0 = time.time()
    run_inference(model_path=MODEL_DIR, input_path=SEGY_PATH, output_path=tmp, direction=DIRECTION,
                  densify=DENSIFY, num_overlap_patches=7, overlap_filter="ramp", batch_size=NCS_BATCH,
                  device="cuda" if torch.cuda.is_available() else "cpu", dtype=NCS_DTYPE)
    print(f"inference took {(time.time()-t0)/60:.1f} min")
    shutil.copytree(tmp, FEAT_PATH); print("saved to", FEAT_PATH)

## 4. Load and check registration

The NCS coordinates are converted to inline/crossline numbers and TWT. Check that they fall inside the seismic and sit where you'd expect. As a visual check, the first principal component is drawn over the seismic in section 5. Its boundaries should line up with the seismic.

In [ ]:
ds = xr.open_zarr(FEAT_PATH, group=DIRECTION)
F = ds["features"]
print(ds)
print()
print(N.describe_registration(ds, ilines, xlines, samples))
MAPS = tuple(N.axis_maps(ds[a].values, n) for a, n in (("inline", NI), ("xline", NX), ("time_depth", NT)))
fi = int(MAPS[0][1][ci]); fx = int(MAPS[1][1][cx])                     # nearest feature trace to the well
ft = int(MAPS[2][1][int(np.clip(round((WELL_TWT_MS - samples[0]) / dt), 0, NT - 1))])
print(f"\nwell reference → feature grid index (inline {fi}, xline {fx}, time {ft}) = "
      f"inline {ilines[ds.inline.values[fi]]}, crossline {xlines[ds.xline.values[fx]]}, {samples[ds.time_depth.values[ft]]:.0f} ms")

## 5. PCA, and the first three components on the seismic

In [ ]:
from sklearn.decomposition import PCA
t0 = time.time()
Xs = N.sample_vectors(F, PCA_SAMPLES, seed=SEED, normalise=L2_NORMALISE)
pca = PCA(n_components=N_PCA, svd_solver="randomized", random_state=SEED).fit(Xs)
P = N.pca_transform(F, pca, normalise=L2_NORMALISE)                     # (il_f, xl_f, t_f, N_PCA)
cum = np.cumsum(pca.explained_variance_ratio_)
print(f"PCA on {len(Xs):,} vectors: {time.time()-t0:.0f} s; cumulative variance at 3/8/16/32 PCs: "
      + ", ".join(f"{cum[k-1]*100:.1f}%" for k in (3, 8, 16, 32) if k <= N_PCA))
np.save(os.path.join(OUT_DIR, f"{stem}_ncs_pca{N_PCA}_featuregrid.npy"), P.astype(np.float16))

def rgb3(Pf, idx):
    rgb = np.stack([N.to_seismic_grid(Pf[..., k], MAPS)[idx] for k in range(3)], -1)
    lo, hi = np.percentile(rgb, [2, 98], axis=(0, 1))
    return np.clip((rgb - lo) / (hi - lo + 1e-9), 0, 1)

EXT_IL = (xlines[0], xlines[-1], samples[-1], samples[0])
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
axes[0].imshow(cube[ci].T, cmap="gray", aspect="auto", vmin=-vclip, vmax=vclip, extent=EXT_IL); axes[0].set_title(f"seismic, inline {ilines[ci]}")
rgb = rgb3(P, (ci, slice(None), slice(None)))
axes[1].imshow(rgb.transpose(1, 0, 2), aspect="auto", extent=EXT_IL); axes[1].set_title("NCS PC1–3 as RGB")
axes[2].imshow(cube[ci].T, cmap="gray", aspect="auto", vmin=-vclip, vmax=vclip, extent=EXT_IL)
axes[2].imshow(rgb.transpose(1, 0, 2), aspect="auto", extent=EXT_IL, alpha=0.45); axes[2].set_title("overlay (registration check)")
for ax in axes: ax.set_xlabel("crossline"); ax.set_ylabel("TWT (ms)")
fig.tight_layout(); plt.show()
plt.figure(figsize=(6, 3.5)); plt.plot(np.arange(1, N_PCA + 1), cum * 100, "o-"); plt.xlabel("PCs"); plt.ylabel("cumulative variance (%)"); plt.grid(alpha=.3); plt.show()

## 6. K-means facies for several K

These are clusters of seismic character, **not** geological facies until checked against the seismic, the wells and your geology. Cluster numbers are arbitrary.

In [ ]:
from sklearn.cluster import MiniBatchKMeans
LAB = {}
flatP = P.reshape(-1, N_PCA)
rng = np.random.default_rng(SEED); fit_idx = rng.choice(len(flatP), min(250_000, len(flatP)), replace=False)
for K in K_VALUES:
    km = MiniBatchKMeans(n_clusters=K, random_state=SEED, n_init=5, batch_size=10_000).fit(flatP[fit_idx])
    LAB[K] = km.predict(flatP).reshape(P.shape[:3]).astype(np.int16)
    pops = np.bincount(LAB[K].ravel(), minlength=K) / LAB[K].size * 100
    print(f"K={K:2d}: populations " + " ".join(f"{p:.0f}%" for p in pops))

fig, axes = plt.subplots(2, len(K_VALUES), figsize=(5.5 * len(K_VALUES), 10))
k_t = int(np.clip(round((sum(INTERVAL_MS) / 2 - samples[0]) / dt), 0, NT - 1))
for col, K in enumerate(K_VALUES):
    lab_g_sec = N.to_seismic_grid(LAB[K], MAPS, order=0)
    axes[0, col].imshow(cube[ci].T, cmap="gray", aspect="auto", vmin=-vclip, vmax=vclip, extent=EXT_IL)
    axes[0, col].imshow(lab_g_sec[ci].T, cmap="tab20", aspect="auto", extent=EXT_IL, alpha=0.5, vmin=0, vmax=19, interpolation="nearest")
    axes[0, col].set_title(f"K={K}: inline {ilines[ci]}"); axes[0, col].set_ylabel("TWT (ms)")
    axes[1, col].imshow(lab_g_sec[:, :, k_t], cmap="tab20", origin="lower", aspect="auto", vmin=0, vmax=19, interpolation="nearest",
                        extent=(xlines[0], xlines[-1], ilines[0], ilines[-1]))
    axes[1, col].plot(xlines[cx], ilines[ci], "k*", ms=12); axes[1, col].set_title(f"K={K}: time slice {samples[k_t]:.0f} ms")
fig.tight_layout(); plt.show()

## 7. Similarity to the well

Cosine similarity of every embedding to the one at the well (`WELL_IL`, `WELL_XL`, `WELL_TWT_MS`). Where it's high, the seismic looks like the rock at the well.
In the LFM this could weight how much the well counts at each point **at the same RGT level**. Here it's just shown, not yet used.

In [ ]:
ref = np.asarray(F.isel(inline=fi, xline=fx, time_depth=ft).values, np.float32)
SIM = N.cosine_to(F, ref)
sim_g = N.to_seismic_grid(SIM, MAPS)
fig, axes = plt.subplots(1, 2, figsize=(20, 6), gridspec_kw={"width_ratios": [2, 1.2]})
axes[0].imshow(cube[ci].T, cmap="gray", aspect="auto", vmin=-vclip, vmax=vclip, extent=EXT_IL)
im = axes[0].imshow(sim_g[ci].T, cmap="magma", aspect="auto", extent=EXT_IL, alpha=0.6)
axes[0].plot(xlines[cx], samples[ds.time_depth.values[ft]], "c*", ms=16); axes[0].set_title(f"similarity to the well embedding, inline {ilines[ci]}")
plt.colorbar(im, ax=axes[0])
k_ref = int(ds.time_depth.values[ft])
im = axes[1].imshow(sim_g[:, :, k_ref], cmap="magma", origin="lower", aspect="auto", extent=(xlines[0], xlines[-1], ilines[0], ilines[-1]))
axes[1].plot(xlines[cx], ilines[ci], "c*", ms=14); axes[1].set_title(f"similarity at {samples[k_ref]:.0f} ms"); plt.colorbar(im, ax=axes[1])
fig.tight_layout(); plt.show()

## 8. Lateral embedding similarity vs amplitude coherence

How similar each embedding is to its sideways neighbours (low = abrupt change: fault, facies boundary or chaos), next to the amplitude-based coherence the classical RGT uses to cut dip links.
If the embedding version picks out the chaotic zone below Ranger more clearly, it's a candidate for improving the classical/hybrid RGT.

In [ ]:
import rgt_classic as C; importlib.reload(C)
t0 = time.time()
s_il, s_xl = N.lateral_similarity(F)
LATSIM = N.to_seismic_grid(N.node_from_links(s_il, s_xl), MAPS)
p_, q_, qual_, LINKS = C.fault_aware_dips(cube)
COH = C.voxel_weight_from_links(LINKS, power=1)
print(f"computed in {time.time()-t0:.0f} s")
fig, axes = plt.subplots(2, 2, figsize=(20, 11), gridspec_kw={"width_ratios": [2, 1.2]})
for row, (vol, name) in enumerate(((COH, "amplitude coherence (classical RGT)"), (LATSIM, "NCS lateral embedding similarity"))):
    lo, hi = np.percentile(vol, [2, 99.5])
    axes[row, 0].imshow(vol[ci].T, cmap="gray", aspect="auto", vmin=lo, vmax=hi, extent=EXT_IL)
    axes[row, 0].set_title(f"{name}, inline {ilines[ci]}"); axes[row, 0].set_ylabel("TWT (ms)")
    axes[row, 1].imshow(vol[:, :, k_t], cmap="gray", origin="lower", aspect="auto", vmin=lo, vmax=hi,
                        extent=(xlines[0], xlines[-1], ilines[0], ilines[-1]))
    axes[row, 1].set_title(f"{name}, {samples[k_t]:.0f} ms")
fig.tight_layout(); plt.show()
r = np.corrcoef(COH[:, :, ::4].ravel(), LATSIM[:, :, ::4].ravel())[0, 1]
print(f"correlation between the two discontinuity measures: {r:.2f}")

## 9. Save on the seismic grid (SEG-Y, headers copied from the input)

In [ ]:
def write_like_input(volume, tag):
    path = os.path.join(OUT_DIR, f"{stem}_{tag}.sgy")
    shutil.copyfile(SEGY_PATH, path)
    with segyio.open(path, "r+", ignore_geometry=True) as f:
        for k in range(G["n_tr"]):
            f.trace[k] = np.ascontiguousarray(volume[G["il_idx"][k], G["xl_idx"][k]], dtype=np.float32)
    print("wrote", os.path.basename(path))
for K in K_VALUES:
    write_like_input(N.to_seismic_grid(LAB[K], MAPS, order=0).astype(np.float32), f"ncs_facies_K{K}")
for k in range(3):
    write_like_input(N.to_seismic_grid(P[..., k], MAPS), f"ncs_PC{k+1}")
write_like_input(sim_g, f"ncs_similarity_to_well_{WELL_TWT_MS}ms")
write_like_input(LATSIM, "ncs_lateral_similarity")
print("all outputs in", OUT_DIR)